In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")
PHENOMETRICS_FILE = RAW_DATA_DIR / "climate_matched_phenometrics.geojson"
phenometrics = gpd.read_file(PHENOMETRICS_FILE)
pd.set_option("display.max_columns", None)
phenometrics.head(1)


In [ ]:
# Strip columns from census data
phenometrics = phenometrics.drop(columns=["index_right", "COUNTYNS", "STATEFP", "COUNTYFP", "AFFGEOID", "GEOID", "NAME", "NAMELSAD", "STUSPS", "STATE_NAME", "LSAD", "ALAND", "AWATER"])
phenometrics.head(1)


In [ ]:
# Create list of DOY active for each row
phenometrics["Days_Active"] = [ list(range(phenometrics["First_Yes_Julian_Date"][row], phenometrics["Last_Yes_Julian_Date"][row] + 1)) for row in range(len(phenometrics))]
phenometrics['Days_Active_DOY'] = phenometrics['Days_Active'].apply(
    lambda dates_list: pd.to_datetime(dates_list, origin='julian', unit='D').dayofyear.tolist()
)



In [ ]:
# Make dataframe show individual rows for each active day
phenometrics = phenometrics.explode("Days_Active_DOY")

In [ ]:
# Create dataframe showing how many of each species exhibit a phenophase on a specific DOY
species_activity = phenometrics.groupby(["scientific_name", "Days_Active_DOY", "Phenophase_ID"])["Individual_ID"].nunique().reset_index(name="Active_Count")

In [ ]:
total_count_per_species = phenometrics.groupby("scientific_name")["Individual_ID"].nunique().reset_index(name="Num_Species")
sorted_count = total_count_per_species.sort_values(by="Num_Species", ascending=False)
sorted_count.head(50)

In [ ]:
# Add column for number of species to each row
species_activity = pd.merge(species_activity, sorted_count, on="scientific_name", how="inner")

In [ ]:
# Percent showing the phenophase per each day
species_activity["Percent_Active"] = (species_activity["Active_Count"] / species_activity["Num_Species"]) * 100
species_activity

In [ ]:
# Create df showing phenophase activity on each DOY
genus_activity = phenometrics.groupby(["Phenophase_ID", "Genus", "Days_Active_DOY"])["Individual_ID"].nunique().reset_index(name="Active_Count")
genus_count = phenometrics.groupby("Genus")["Individual_ID"].nunique().reset_index(name="Num_Genus")
genus_activity = pd.merge(genus_activity, genus_count, on="Genus", how="inner")
genus_activity["Percent_Active"] = (genus_activity["Active_Count"] / genus_activity["Num_Genus"]) * 100
genus_activity

In [ ]:
# Create pivot tables for each year describing activity on each day
full_year = list(range(1,367))
genus_activity_table = pd.pivot_table(
 data=genus_activity, index=["Genus", "Phenophase_ID"], columns= "Days_Active_DOY", values= "Percent_Active", fill_value=0   
)
genus_activity_table = genus_activity_table.reindex(columns=full_year, fill_value=0)
genus_activity_table.columns.name = None

species_activity_table = pd.pivot_table(
    data=species_activity, index=["scientific_name", "Phenophase_ID"], columns="Days_Active_DOY", values="Percent_Active", fill_value=0
)
species_activity_table = species_activity_table.reindex(columns=full_year, fill_value=0)
species_activity_table.columns.name = None


In [ ]:
sorted_genus_count = genus_count.sort_values(by="Num_Genus", ascending=False)
sorted_genus_count.head(50)

In [ ]:
sorted_count.head(50)

In [ ]:
melt_practice = species_activity.melt()
melt_practice